In [6]:
import requests
import pandas as pd
import time
import os
from datetime import datetime
from dotenv import load_dotenv

I dont think that this sould be a notebook but that's a fix for later!

# 1. Parameters

# 2. Token generation

In [7]:
load_dotenv()
CLIENT_ID = os.getenv('SH_ID') 
CLIENT_SECRET = os.getenv('SH_SECRET') 

def get_token():
    auth_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    r = requests.post(auth_url, data={"grant_type": "client_credentials", "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET})
    r.raise_for_status()
    return r.json().get("access_token")

# 3. Grid generation

In [8]:
def generate_grid(bbox, krok):
    min_lon, min_lat, max_lon, max_lat = bbox
    sectors = []
    idx = 1
    curr_lon = min_lon
    while curr_lon < max_lon:
        curr_lat = min_lat
        while curr_lat < max_lat:
            sectors.append({
                "ID": f"S_{idx}",
                "bbox": [curr_lon, curr_lat, curr_lon + krok, curr_lat + krok],
                "lat": round(curr_lat + (krok/2), 5),
                "lon": round(curr_lon + (krok/2), 5)
            })
            idx += 1
            curr_lat += krok
        curr_lon += krok
    return sectors


# 3. Data download

In [11]:
def round_fr(wartosc, miejsca_po_przecinku=4):
    try:
        return round(float(wartosc), miejsca_po_przecinku)
    except (ValueError, TypeError):
        return None

In [12]:
def get_data(OUTPUT_FILE, BBOX, SECTOR_SIZE):

    token = get_token()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    url = "https://sh.dataspace.copernicus.eu/api/v1/statistics"
    
    sectors = generate_grid(BBOX, SECTOR_SIZE)
    
    evalscript = """
    //VERSION=3
    function setup() {
      return {
        input: ["B03", "B04", "B08", "B11", "SCL", "dataMask"],
        output: [
          { id: "ndvi", bands: 1 },
          { id: "ndwi", bands: 1 },
          { id: "ndmi", bands: 1 },
          { id: "dataMask", bands: 1 } // <--- TO JEST KLUCZOWE
        ]
      };
    }
    function evaluatePixel(samples) {
      // Ignorujemy chmury i cienie
      if ([3, 8, 9, 10].includes(samples.SCL) || samples.dataMask === 0) {
          return { ndvi: [NaN], ndwi: [NaN], ndmi: [NaN], dataMask: [0] };
      }
      let ndvi = (samples.B08 - samples.B04) / (samples.B08 + samples.B04);
      let ndwi = (samples.B03 - samples.B08) / (samples.B03 + samples.B08);
      let ndmi = (samples.B08 - samples.B11) / (samples.B08 + samples.B11);
      
      // Zwracamy wszystkie parametry + potwierdzenie, że piksel jest prawidłowy (1)
      return { ndvi: [ndvi], ndwi: [ndwi], ndmi: [ndmi], dataMask: [1] };
    }
    """
    all_data = []

    for i, s in enumerate(sectors):
        print(f"{i+1}/{len(sectors)}: {s['ID']} (Wrocław)...", end="\r")
        payload = {
            "input": {
                "bounds": {"bbox": s['bbox']},
                "data": [{"type": "sentinel-2-l2a", "dataFilter": {"mosaickingOrder": "leastCC"}}]
            },
            "aggregation": {
                "timeRange": {"from": "2025-06-01T00:00:00Z", "to": "2026-04-20T00:00:00Z"},
                "aggregationInterval": {"of": "P5D"},
                "evalscript": evalscript,
                "resx": 0.0005, "resy": 0.0005
            }
        }

        res = requests.post(url, headers=headers, json=payload)
        if res.status_code == 200:
            data = res.json()
            for item in data['data']:
                data_iso = item['interval']['from'][:10]
                
                out = item['outputs']
                if out['ndvi']['bands']['B0']['stats']['sampleCount'] > 0:
                    all_data.append({
                        "Sektor_ID": s['ID'],
                        "Lat": s['lat'],
                        "Lon": s['lon'],
                        "Data": data_iso,
                        "NDVI": round_fr(out['ndvi']['bands']['B0']['stats']['mean'], 4),
                        "NDWI": round_fr(out['ndwi']['bands']['B0']['stats']['mean'], 4),
                        "NDMI": round_fr(out['ndmi']['bands']['B0']['stats']['mean'], 4)
                    })
        time.sleep(0.4) # Bezpiecznik limitów

    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✨ Success! Results saved in {OUTPUT_FILE}")

In [13]:
DATA_FOLDER = os.path.join("..", "data")
if not os.path.exists(DATA_FOLDER):
    os.makedirs(DATA_FOLDER)

OUTPUT_FILE = os.path.join(DATA_FOLDER, "wroclaw_data.csv")

BBOX = [16.80, 51.02, 16.80+0.1, 51.02+0.1] # mniejsza wersja
#WROCLAW_BBOX = [16.80, 51.02, 17.17, 51.20] 
SECTOR_SIZE = 0.02 # ~ 2km

get_data(OUTPUT_FILE, BBOX, SECTOR_SIZE)

30/30: S_30 (Wrocław)...
✨ Success! Results saved in ..\data\wroclaw_data.csv


# 

In [14]:

df = pd.read_csv("../data/wroclaw_data.csv")

# Upewniamy się, że kolumna 'Data' to format czasu, a dane są posortowane!
df['Data'] = pd.to_datetime(df['Data'])
df = df.sort_values(by=["Sektor_ID", "Data"]).reset_index(drop=True)

# 2. Definiujemy, które kolumny chcemy naprawić
wskazniki = ['NDVI', 'NDWI', 'NDMI']

# KROK A: Uzupełnianie braków (Interpolacja liniowa)
# Używamy groupby, żeby nie połączyć przypadkiem 30 czerwca z Sektora_1 z 1 lipca z Sektora_2!
print("⏳ Łatanie dziur po chmurach...")
df[wskazniki] = df.groupby('Sektor_ID')[wskazniki].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

# KROK B: Wygładzanie (Średnia krocząca) - redukcja szumów
print("🌊 Wygładzanie anomalii...")
wielkosc_okna = 3
df[wskazniki] = df.groupby('Sektor_ID')[wskazniki].transform(
    lambda x: x.rolling(window=wielkosc_okna, min_periods=1, center=True).mean()
)

# KROK C: Eleganckie zaokrąglenie do 4 miejsc po przecinku
df[wskazniki] = df[wskazniki].round(4)

print("\n✅ Gotowe! Tak wyglądają Twoje wyczyszczone dane dla Sektora 1:")
display(df.head(10))

# Opcjonalnie: Zapisz wyczyszczoną wersję do nowego pliku
df.to_csv("../data/wroclaw_clean_data.csv", index=False)

⏳ Łatanie dziur po chmurach...
🌊 Wygładzanie anomalii...

✅ Gotowe! Tak wyglądają Twoje wyczyszczone dane dla Sektora 1:


,Sektor_ID,Lat,Lon,Data,NDVI,NDWI,NDMI
0,S_1,51.03,16.81,2025-06-01,0.7103,-0.6495,0.4341
1,S_1,51.03,16.81,2025-06-06,0.7078,-0.6486,0.4280
2,S_1,51.03,16.81,2025-06-11,0.6557,-0.6204,0.3883
3,S_1,51.03,16.81,2025-06-16,0.6102,-0.5983,0.3506
4,S_1,51.03,16.81,2025-06-21,0.5710,-0.5823,0.3150
5,S_1,51.03,16.81,2025-06-26,0.5484,-0.5534,0.3149
6,S_1,51.03,16.81,2025-07-01,0.5006,-0.5262,0.2634
7,S_1,51.03,16.81,2025-07-06,0.4192,-0.4724,0.1755
8,S_1,51.03,16.81,2025-07-11,0.3408,-0.4376,0.0455
9,S_1,51.03,16.81,2025-07-16,0.3025,-0.3826,0.0286
